In [2]:
import os
import pdfplumber
import pytesseract as pt
from pdf2image import convert_from_path
from autocorrect import Speller
import re
import pandas as pd
from datetime import datetime as dt

In [3]:
file = '../documents/990Ts/990T_2021.pdf'

In [4]:
def process_investments(page_img):
    text = pt.image_to_string(page_img)
    details = pd.Series([item for item in text.split('\n') if re.search(r'^\([0-9]{1}\)', item)])
    return {k:v for k, v in zip(details.apply(lambda x: re.sub(r'^\([0-9]{1}\)', '', x.split(':')[0]).strip()), details.apply(lambda x: x.split(':', 1)[1].strip()))}

def get_investments_singular(file, page_num):
    page = convert_from_path(file, 600, first_page=page_num, last_page=page_num)[0]
    return pd.DataFrame(process_investments(page), index=[0])

def get_investments_multiple(file, first, last):
    pages = convert_from_path(file, 600, first_page=first, last_page=last)
    return pd.concat([pd.DataFrame(process_investments(page), index=[0]) for page in pages])

def get_page_text(file, page_num):
    page = convert_from_path(file, 600, first_page=page_num, last_page=page_num)[0]
    return pt.image_to_string(page)

In [5]:
def format_date(x):
    try:
        date = dt.strptime(x, '%m/%d/%Y')
    except:
        date = None
    return date

In [85]:
# start page 17
# end page 93
df = get_investments_multiple(file, 17, 93)
df.to_csv('../data/stock-transfers_2021_raw.csv', index=False)

In [14]:
def listify_stock_page(file, page_num):
    text = get_page_text(file, page_num)
    return list(filter(lambda x: len(x) > 0, text.split('\n')))

In [11]:
splits = [
    '\(1\) Name of Transferor\:',
    'EIN\:',
    'Address\:',
    '\(2\) Name of Transferee\:',
    'EIN:',
    'Address:',
    'Country of Incorporation',
    '\(3\) Consideration received:',
    '\(4\) Cash'
]

def clean_stock_frame(df):
    df.columns = [
        'Transferor',
        'Transferor_EIN',
        'Transferor_Address',
        'Transferee',
        'Transferee_EIN',
        'Transferee_Address',
        'Transferee_Country_Incorporation',
        'Consideration_Received',
        'Cash'
    ]

    df['Description'] = df['Transferee_Country_Incorporation'].apply(lambda x: x.split('\n\n')[1] if len(x.split('\n\n')) > 1 else None)
    df['Transferee_Country_Incorporation'] = df['Transferee_Country_Incorporation'].apply(lambda x: x.split('\n\n')[0])

    df['Footer'] = df['Cash'].apply(lambda x: x.split('\n\n')[1] if len(x.split('\n\n')) > 1 else None)
    df['Cash'] = df['Cash'].apply(lambda x: x.split('\n\n')[0])

    return df

def extract_stocks(file, page_num):
    text = get_page_text(file, page_num)
    
    curr_term = 0
    result = {}
    while len(text) > 0 and curr_term < len(splits):
        start = re.search(splits[curr_term], text).start(0)
        text = text[start:]
        out_key = splits[curr_term]
        curr_term += 1
        if curr_term < len(splits):
            end = re.search(splits[curr_term], text).start(0)
            result[f'{curr_term}_{out_key}'] = text[len(out_key.replace("\\",'')):end].strip()
            text = text[end:]
        else:
            result[f'{curr_term}_{out_key}'] = text[len(out_key.replace("\\",'')):].strip()

    df = pd.DataFrame(result, index=[0])
    df = clean_stock_frame(df)
    df['Page'] = page_num

    return df

In [6]:
def regex_extract_stocks(file, page_num):
    text = get_page_text(file, page_num)
    return pd.DataFrame({
        'transferee': text[re.search(r'44074', text).end(0):re.search(r'Oberlin College \(EIN', text).start(0)],
        'description': re.search(r'(Oberlin College \(EIN.*)Ownership', text, re.DOTALL).group(1),
        'consideration_received': re.search(r'Ownership.*corporation', text, re.DOTALL).group(0),
        'amount':re.search(r'\$[0-9,]+', text).group(0),
        'page': page_num
    }, index=[page_num])

In [7]:
df = pd.concat([regex_extract_stocks(file, page) for page in range(94, 171)])
df.to_csv('../data/investments_2021_raw.csv', index=False)

In [60]:
df = pd.read_csv('../data/investments_2021_raw.csv')

In [61]:
df.reset_index(drop=True, inplace=True)

In [62]:
df['transferee'] = df['transferee'].apply(lambda x: x.replace('(2) Name of Transferee:','').replace('\n\n','\n').strip())

In [63]:
df['transferee_name'] = df['transferee'].apply(lambda x: x.split('\n')[0])

In [64]:
df['transferee_ein'] = df['transferee'].apply(lambda x: x.split('\n')[1])
df['transferee_ein'] = df['transferee_ein'].apply(lambda x: x.replace('EIN:', '').strip())

In [65]:
df['transferee_country'] = df['transferee'].apply(lambda x: x.split('\n')[-1])
df['transferee_country'] = df['transferee_country'].apply(lambda x: x.replace('Country of Incorporation', '').strip())

In [66]:
df['transferee_address'] = df['transferee'].apply(lambda x: ' '.join(x.split('\n')[2:-2]))
df['transferee_address'] = df['transferee_address'].apply(lambda x: x.replace('Address:', '').strip())

In [67]:
df['overseer_name'] = df['description'].apply(lambda x: re.search(r'owns an interest in (.*)\(EIN', x, re.DOTALL).group(1).replace('\n',' ').strip())
df['overseer_ein'] = df['description'].apply(lambda x: re.search(r'owns an interest in .*EIN: ([0-9]+\-[0-9]+)', x, re.DOTALL).group(1) if re.search(r'owns an interest in .*EIN: ([0-9]+\-[0-9]+)', x, re.DOTALL) is not None else '')
df['overseer_name'] = df['overseer_name'].apply(lambda x: x.strip())
df['overseer_ein'] = df['overseer_ein'].apply(lambda x: x.replace('),','').strip())

In [68]:
df['consideration_received'] = df['consideration_received'].apply(lambda x: ' '.join(x.split(' ')[0:2]))

In [69]:
df['amount'] = df['amount'].apply(lambda x: pd.to_numeric(x.replace('$','').replace(',','')))

In [70]:
df['transferee_ein'] = df['transferee_ein'].replace('N/A', None)

In [71]:
df = df[['transferee_name', 'overseer_name', 'amount', 'transferee_country', 'transferee_address', 'transferee_ein', 'overseer_ein', 'page']]

In [72]:
df.to_csv('../data/investments_2021_clean.csv', index=False)